# Analysis: EDA, verification of results, explanations
Run **after** `run_all.ipynb` (needs `vit.pt`, `cnn.pt` and `results.json` in Drive). GPU runtime. Outputs go to the repo folders `EDA/`, `figures/`, `Results/` inside Colab, are copied to Drive, and zipped at the end so you can drop them into your local repo and push.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/xai-cnn-vit'
import os, sys
REPO = 'https://github.com/hsfcoulibaly/Explanability-Methods.git'
if not os.path.exists('/content/repo'):
    !git clone -q $REPO /content/repo
else:
    !cd /content/repo && git pull -q
ROOT = '/content/repo'; PROJECT = ROOT + '/ViT'
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
!pip install -q timm captum scipy pandas
FIG, RES, EDA = ROOT + '/figures', ROOT + '/Results', ROOT + '/EDA'
SCORES = CKPT_DIR + '/scores'
for d in (FIG, RES, EDA, SCORES): os.makedirs(d, exist_ok=True)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## A. EDA
Dataset facts, class balance, whether the 1,000-image eval subset is representative, why heatmaps look blocky (32px upscaled to 224), and per-class accuracy of both models.

In [ ]:
!python {EDA}/eda.py
!python -m src.verify perclass --cnn-ckpt $CKPT_DIR/cnn.pt --vit-ckpt $CKPT_DIR/vit.pt --out {EDA}/per_class_accuracy.json --fig {FIG}/eda_per_class_accuracy.png

## B. Verification of results
1. Per-image scores for every method, on the real map and the **inverted** map (control).
2. Bootstrap 95% CIs, paired difference vs random, correct-vs-wrong split, method agreement.

In [ ]:
for arch in ['cnn', 'vit']:
    !python -m src.verify collect --arch {arch} --ckpt $CKPT_DIR/{arch}.pt --out-dir {SCORES}

In [ ]:
!python -m src.verify summarize --scores-dir {SCORES} --fig-dir {FIG} --res-dir {RES}

Randomization sanity check (cascading, top-down): maps should decorrelate from the original as layers are re-initialized.

In [ ]:
for arch in ['vit', 'cnn']:
    !python -m src.sanity --arch {arch} --ckpt $CKPT_DIR/{arch}.pt --out {RES}/sanity_{arch}.json --n-images 64
import json, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k, arch in enumerate(['cnn', 'vit']):
    for m, v in json.load(open(f'{RES}/sanity_{arch}.json')).items():
        ax[k].plot(list(v.keys()), list(v.values()), marker='o', label=m)
    ax[k].set_title(f'{arch.upper()}: Spearman vs original map as layers are randomized', fontsize=9)
    ax[k].tick_params(axis='x', rotation=60); ax[k].legend(); ax[k].axhline(0, color='k', lw=.8)
plt.tight_layout(); plt.savefig(f'{FIG}/sanity_randomization.png', dpi=150); plt.show()

## C. Explanations: why is ViT Grad-CAM weak?
Tries different target layers for Grad-CAM on the ViT. If another layer scores well, the weakness is a hook-choice effect, not Grad-CAM as a whole.

In [ ]:
!python -m src.verify hooks --ckpt $CKPT_DIR/vit.pt --out {RES}/gradcam_hook_ablation.json --fig {FIG}/gradcam_hook_ablation.png

Qualitative: same images, all methods, where the model is right vs wrong.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.models import load_checkpoint
from src.data import get_eval_subset, denormalize
from src.explain import get_explainer
device = 'cuda'
x, y = next(iter(get_eval_subset(batch_size=16)))
x8 = x[:8].to(device)
rows = [('cnn', 'gradcam'), ('cnn', 'ig'), ('vit', 'gradcam'), ('vit', 'ig'), ('vit', 'rollout')]
fig, ax = plt.subplots(1 + len(rows), 8, figsize=(16, 2.1 * (1 + len(rows))))
preds = {}
for r, (arch, m) in enumerate(rows, 1):
    model, _ = load_checkpoint(arch, f'{CKPT_DIR}/{arch}.pt', device)
    s, p = get_explainer(m)(model, x8); preds[arch] = p.cpu()
    for i in range(8):
        ax[r, i].imshow(denormalize(x8[i]).permute(1,2,0).cpu()); ax[r, i].imshow(s[i].cpu(), cmap='jet', alpha=.5)
    ax[r, 0].set_ylabel(f'{arch} {m}', fontsize=8)
for i in range(8):
    ax[0, i].imshow(denormalize(x8[i]).permute(1,2,0).cpu())
    ax[0, i].set_title(f"y={y[i].item()} cnn={preds['cnn'][i].item()} vit={preds['vit'][i].item()}", fontsize=7)
for a in ax.flat: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(f'{FIG}/qualitative_all_methods.png', dpi=130); plt.show()

## D. Save everything
Copies outputs to Drive and zips them. Download the zip, unzip it at the root of your local repo (folders `EDA/`, `figures/`, `Results/` match), then commit and push.

In [ ]:
import shutil
for d in ('EDA', 'figures', 'Results'):
    shutil.copytree(f'{ROOT}/{d}', f'{CKPT_DIR}/outputs/{d}', dirs_exist_ok=True)
import zipfile
with zipfile.ZipFile('/content/analysis_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for d in ('EDA', 'figures', 'Results'):
        for dp, _, fs in os.walk(f'{ROOT}/{d}'):
            for f in fs:
                p = os.path.join(dp, f); z.write(p, os.path.relpath(p, ROOT))
shutil.copy('/content/analysis_outputs.zip', f'{CKPT_DIR}/analysis_outputs.zip')
from google.colab import files; files.download('/content/analysis_outputs.zip')